# Prompt Evaluation & Optimization with Vertex AI SDK

# ----------------------------------------------------

This notebook demonstrates how to perform prompt evaluation and optimization
using Vertex AI Generative AI Evaluation SDK.
It includes: dataset prep, metric setup, evaluation run, optimization loop.

## Setup and installation


In [1]:
!pip install --upgrade google-cloud-aiplatform[evaluation] --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.30.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.


In [1]:
import os
import pandas as pd
from google.cloud import aiplatform
import vertexai
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate

In [2]:
PROJECT_ID = "bdc-trainings"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Initialized Vertex AI for project {PROJECT_ID}, location {LOCATION}")

Initialized Vertex AI for project bdc-trainings, location us-central1


## Prepare evaluation dataset


In [3]:
df = pd.DataFrame({
    'prompt': ["Summarize the benefits of AI in healthcare.", "Explain cloud computing in simple terms."],
    'response': ["AI improves diagnostics and personalized care.", "Cloud computing means using remote servers for storage and processing."],
    'reference': ["AI in healthcare aids diagnosis and treatment personalization.", "Cloud computing uses internet-based servers instead of local ones."]
})

df.to_json("eval_dataset.jsonl", orient="records", lines=True)
print("Evaluation dataset created and saved as eval_dataset.jsonl")


Evaluation dataset created and saved as eval_dataset.jsonl


## Define evaluation metrics


In [4]:
readability_metric = PointwiseMetric(
    metric="readability_grade_level",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "grade_level": (
                "Estimate the U.S. grade-level readability of the response. Lower is simpler, higher is complex."
            ),
            "conciseness": (
                "Evaluate how concise the response is: avoids unnecessary words while preserving meaning."
            )
        },
        rating_rubric={
            "1": "Excellent readability and conciseness.",
            "0": "Average readability and conciseness.",
            "-1": "Poor readability or verbosity."
        },
        input_variables=["prompt", "response", "reference"]
    )
)
print("Custom readability metric defined.")


Custom readability metric defined.


## Run evaluation

In [5]:
eval_task = EvalTask(
    dataset="eval_dataset.jsonl",
    metrics=[readability_metric, "bleu", "rouge_l_sum"],
    experiment="prompt-eval-experiment1-anshu"
)

print("Running evaluation task...")
eval_result = eval_task.evaluate()
print("Evaluation completed.")

Running evaluation task...


Associating projects/456822750436/locations/us-central1/metadataStores/default/contexts/prompt-eval-experiment1-anshu-aa80740e-0a01-4fd4-9ddb-bdb2320685c4 to Experiment: prompt-eval-experiment1-anshu


Computing metrics with a total of 6 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 6/6 [00:04<00:00,  1.41it/s]

All 6 metric requests are successfully computed.
Evaluation Took:4.264058672000033 seconds


Evaluation completed.


## View evaluation results


In [6]:
eval_result.summary_metrics

{'row_count': 2,
 'readability_grade_level/mean': np.float64(1.0),
 'readability_grade_level/std': np.float64(0.0),
 'bleu/mean': np.float64(0.0731709925),
 'bleu/std': np.float64(0.011625772399191908),
 'rouge_l_sum/mean': np.float64(0.29285715),
 'rouge_l_sum/std': np.float64(0.010101515343996672)}

In [8]:
per_instance = eval_result.metrics_table
per_instance.T

,0,1
prompt,Summarize the benefits of AI in healthcare.,Explain cloud computing in simple terms.
response,AI improves diagnostics and personalized care.,Cloud computing means using remote servers for...
reference,AI in healthcare aids diagnosis and treatment ...,Cloud computing uses internet-based servers in...
readability_grade_level/explanation,"The response is extremely concise, directly su...","The response is extremely concise, explaining ..."
readability_grade_level/score,1.0,1.0
bleu/score,0.06495,0.081392
rouge_l_sum/score,0.285714,0.3


## New Prompt Evaluation: Summarization Task

This section demonstrates a new prompt evaluation using a summarization task, where the model generates the responses based on the provided prompts. We will evaluate a broader set of metrics including ROUGE-1, ROUGE-2, additional readability scores (ARI, Flesch-Kincaid), and a custom LLM-as-judge metric for conciseness.

In [9]:
# Create a new dataset for summarization
summarization_df = pd.DataFrame({
    'prompt': [
        "Summarize the key points of this article: 'Artificial intelligence (AI) is intelligence demonstrated by machines, unlike the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of 'intelligent agents': any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals. Colloquially, the term 'artificial intelligence' is often used to describe machines (or computers) that mimic 'cognitive' functions that humans associate with the human mind, such as 'learning' and 'problem-solving'. Such functions are typically associated with other human traits, such as emotion; however, such connections are not necessary for a functional system that produces intelligent behavior.'",
        "Summarize the plot of Romeo and Juliet in one sentence."
    ],
    'reference': [
        "AI is machine intelligence that mimics human cognitive functions like learning and problem-solving, defined as the study of intelligent agents that act to achieve goals.",
        "Romeo and Juliet is a tragedy about two young star-crossed lovers whose deaths ultimately reconcile their feuding families."
    ]
})

summarization_df.to_json("summarization_dataset.jsonl", orient="records", lines=True)
print("Summarization dataset created and saved as summarization_dataset.jsonl")

Summarization dataset created and saved as summarization_dataset.jsonl


In [10]:
# Define LLM-as-judge metric for Conciseness
conciseness_metric = PointwiseMetric(
    metric="llm_conciseness",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "conciseness": (
                "Evaluate how concise the response is, avoiding unnecessary words while preserving meaning."
            )
        },
        rating_rubric={
            "1": "Excellent conciseness. The response is brief, to the point, and doesn't contain any superfluous information.",
            "0": "Average conciseness. The response is generally concise but could be slightly shorter without losing meaning, or has minor redundancy.",
            "-1": "Poor conciseness. The response is verbose, contains irrelevant information, or could be significantly shortened."
        },
        input_variables=["prompt", "response", "reference"]
    )
)
print("Custom conciseness metric defined.")

Custom conciseness metric defined.


In [11]:
# Run evaluation for summarization task, with model generating responses
summarization_eval_task = EvalTask(
    dataset="summarization_dataset.jsonl",
    metrics=[
        "bleu",
        "rouge_1",
        "rouge_2",
        "rouge_l_sum",
        conciseness_metric
    ],
    experiment="summarization-eval-experiment"
)

print("Running summarization evaluation task...")
summarization_eval_result = summarization_eval_task.evaluate(
    model="gemini-2.5-flash", # Model to generate responses
    prompt_template="{prompt}" # Template for prompt sent to the model
)
print("Summarization evaluation completed.")

Running summarization evaluation task...


Associating projects/456822750436/locations/us-central1/metadataStores/default/contexts/summarization-eval-experiment-dc431756-5489-4b45-a13d-f380ec75d5d4 to Experiment: summarization-eval-experiment


Logging Eval Experiment metadata: {'prompt_template': '{prompt}', 'model_name': 'gemini-2.5-flash'}
Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.
Generating a total of 2 responses from Gemini model gemini-2.5-flash using genai module.


100%|██████████| 2/2 [00:06<00:00,  3.45s/it]

All 2 responses are successfully generated from Gemini model gemini-2.5-flash using genai module.
Multithreaded Batch Inference took: 7.047921375000101 seconds.
Computing metrics with a total of 10 Vertex Gen AI Evaluation Service API requests.



100%|██████████| 10/10 [00:09<00:00,  1.07it/s]

All 10 metric requests are successfully computed.
Evaluation Took:9.320863593000013 seconds


Summarization evaluation completed.


### View Summarization Evaluation Results

In [12]:
print("Summary Metrics for Summarization Task:")
summarization_eval_result.summary_metrics

Summary Metrics for Summarization Task:


{'row_count': 2,
 'bleu/mean': np.float64(0.050895486000000004),
 'bleu/std': np.float64(0.02593909362490066),
 'rouge_1/mean': np.float64(0.370060805),
 'rouge_1/std': np.float64(0.018268718874854078),
 'rouge_2/mean': np.float64(0.184848485),
 'rouge_2/std': np.float64(0.07285341672661654),
 'rouge_l_sum/mean': np.float64(0.284954415),
 'rouge_l_sum/std': np.float64(0.10208989210775986),
 'llm_conciseness/mean': np.float64(1.0),
 'llm_conciseness/std': np.float64(0.0)}

In [13]:
print("Per-instance Metrics for Summarization Task:")
summarization_per_instance = summarization_eval_result.metrics_table
display(summarization_per_instance)

Per-instance Metrics for Summarization Task:


,prompt,reference,response,bleu/score,rouge_1/score,rouge_2/score,rouge_l_sum/score,llm_conciseness/explanation,llm_conciseness/score
0,Summarize the key points of this article: 'Art...,AI is machine intelligence that mimics human c...,This article explains that Artificial Intellig...,0.069237,0.357143,0.236364,0.357143,"The response is brief, to the point, and effec...",1.0
1,Summarize the plot of Romeo and Juliet in one ...,Romeo and Juliet is a tragedy about two young ...,Two young lovers from feuding families fall in...,0.032554,0.382979,0.133333,0.212766,The response effectively summarizes the entire...,1.0
